In [0]:
# ============================================================
# Medallion Architecture — Bronze → Silver + Quarantine
# sp_certification Pipeline — v2
# ============================================================

# COMMAND ----------
# STEP 1: Import Components and Establish Environment Context

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Medallion_Architecture_Silver_And_Quarantine") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Ensure target database schemas exist in your metastore
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS quarantine")

print("Target database environments ('silver' and 'quarantine') verified.")


# COMMAND ----------
# STEP 2: Ingest Raw Records from the Bronze Delta Table
# (Includes your core features plus: _ingestion_timestamp, _source_system, _source_file)

bronze_table_name = "hackathon_ltm.bronze.sp_certification"
df_bronze = spark.read.table(bronze_table_name)

print(f"Bronze records ingested: {df_bronze.count()}")


# COMMAND ----------
# STEP 3: Silver Layer Transformations (Cleansing & Standardization)

# 3.1 Trim trailing/leading spaces from all string columns dynamically
string_cols = [field.name for field in df_bronze.schema.fields if str(field.dataType) == "StringType()"]
df_transformed = df_bronze
for col_name in string_cols:
    df_transformed = df_transformed.withColumn(col_name, F.trim(F.col(col_name)))

# 3.2 Standardize 'certificate_uploaded' to categorical 'Yes'/'No' binary flags
df_transformed = df_transformed.withColumn(
    "certificate_uploaded_clean",
    F.when(F.upper(F.col("certificate_uploaded")).isin("YES", "Y"), "Yes")
     .when(F.upper(F.col("certificate_uploaded")).isin("NO", "N"), "No")
     .otherwise(F.col("certificate_uploaded"))
)

# 3.3 Force certificate_status to Title Case (e.g. verified → Verified, PENDING → Pending)
df_transformed = df_transformed.withColumn(
    "certificate_status_clean",
    F.initcap(F.col("certificate_status"))
)

# 3.4 Parse string date masks to strongly typed PySpark Date objects
date_mask = "M/d/yyyy"
df_transformed = df_transformed \
    .withColumn("upload_date_parsed",       F.to_date(F.col("upload_date"),       date_mask)) \
    .withColumn("verification_date_parsed", F.to_date(F.col("verification_date"), date_mask))


# COMMAND ----------
# STEP 4: Duplicate Detection & Separation
#
# LOGIC:
#   A duplicate is defined as two or more rows sharing the same
#   (employee_id + course_id) combination.
#
#   Strategy — keep the MOST COMPLETE record, quarantine the rest:
#   - Within each (employee_id + course_id) group, rows are ranked by
#     how complete they are: rows WITH an upload_date come first (rank=1),
#     rows WITHOUT upload_date come after (rank=2+).
#   - Rank 1 → goes forward to silver (the best copy).
#   - Rank 2+ → flagged as duplicates → goes to quarantine.
#
#   This ensures we never silently drop data — every duplicate is
#   preserved in quarantine with a clear error tag for audit.

# Assign a row rank within each (employee_id + course_id) group.
# Rows with upload_date are ranked first (they are more complete).
window_spec = Window \
    .partitionBy("employee_id", "course_id") \
    .orderBy(
        F.col("upload_date_parsed").isNull().asc(),   # False (has date) sorts before True (no date)
        F.col("certification_id").asc()               # tie-break by certification_id for consistency
    )

df_ranked = df_transformed.withColumn("_row_rank", F.row_number().over(window_spec))

# Separate: rank=1 → clean candidate | rank>1 → duplicate to quarantine
df_deduped    = df_ranked.filter(F.col("_row_rank") == 1).drop("_row_rank")
df_duplicates = df_ranked.filter(F.col("_row_rank") > 1).drop("_row_rank")

duplicate_count = df_duplicates.count()
print(f"Duplicate rows detected and separated: {duplicate_count}")


# COMMAND ----------
# STEP 5: Quarantine Rule Evaluation & Constraint Mapping
#
# Rules applied to the DE-DUPLICATED data only:
#
#   RULE 1 — Missing Primary Key
#     certification_id is null or empty string.
#
#   RULE 2 — Corrupted Upload Date
#     upload_date has a value but could not be parsed into a real date.
#
#   RULE 3 — Corrupted Verification Date
#     verification_date has a value but could not be parsed into a real date.
#
#   RULE 4 — Chronological Timeline Violation
#     verification_date is earlier than upload_date — impossible in real life.
#
#   RULE 5 — Uploaded=Yes but upload_date is missing        ← NEW
#     Employee said they uploaded the certificate (Yes/Y)
#     but left no upload date — contradictory record.
#
#   RULE 6 — Status=Verified but verification_date is missing  ← NEW
#     Certificate is marked Verified but there is no date
#     recorded for when it was verified — incomplete record.
#
# NOTE ON NULLS FOR uploaded=No / status=Pending:
#   Rows where uploaded=No and status=Pending will naturally have
#   NULL in both date columns. This is VALID business data
#   (employee hasn't submitted yet) and is NOT flagged as an error.

df_evaluated = df_deduped.withColumn(
    "_failed_constraints",
    F.array(
        # Rule 1 — Missing Primary Key
        F.when(
            F.col("certification_id").isNull() | (F.trim(F.col("certification_id")) == ""),
            "ERR: Missing Primary Key"
        ),

        # Rule 2 — Corrupted Upload Date Format
        F.when(
            F.col("upload_date").isNotNull() & F.col("upload_date_parsed").isNull(),
            "ERR: Corrupted Upload Date Format"
        ),

        # Rule 3 — Corrupted Verification Date Format
        F.when(
            F.col("verification_date").isNotNull() & F.col("verification_date_parsed").isNull(),
            "ERR: Corrupted Verification Date Format"
        ),

        # Rule 4 — Chronological Timeline Violation
        F.when(
            F.col("verification_date_parsed").isNotNull() &
            F.col("upload_date_parsed").isNotNull() &
            (F.col("verification_date_parsed") < F.col("upload_date_parsed")),
            "ERR: Chronological Timeline Violation"
        ),

        # Rule 5 — Uploaded=Yes but upload_date is missing  ← NEW
        F.when(
            (F.col("certificate_uploaded_clean") == "Yes") & F.col("upload_date_parsed").isNull(),
            "ERR: Uploaded=Yes but upload_date is missing"
        ),

        # Rule 6 — Status=Verified but verification_date is missing  ← NEW
        F.when(
            (F.col("certificate_status_clean") == "Verified") & F.col("verification_date_parsed").isNull(),
            "ERR: Status=Verified but verification_date is missing"
        )
    )
)

# Strip NULL elements from the array — only real error strings remain.
# Clean rows → [] (size 0) → Silver
# Dirty rows → ["ERR: ..."] (size > 0) → Quarantine
df_evaluated = df_evaluated.withColumn(
    "_failed_constraints",
    F.array_compact(F.col("_failed_constraints"))
    # Spark < 3.4 alternative:
    # F.expr("filter(_failed_constraints, x -> x is not null)")
)


# COMMAND ----------
# STEP 6: Split Data Streams — Clean vs. Constraint-Failed vs. Duplicates

# 6.1 Rows that failed one or more constraint rules
df_quarantine_constraints = df_evaluated.filter(F.size(F.col("_failed_constraints")) > 0)

# 6.2 Clean silver records — passed all constraint checks
df_silver_batch = (
    df_evaluated
    .filter(F.size(F.col("_failed_constraints")) == 0)
    .drop(
        "certificate_uploaded",
        "certificate_status",
        "upload_date",
        "verification_date",
        "_failed_constraints"
    )
    .withColumnRenamed("certificate_uploaded_clean",   "certificate_uploaded")
    .withColumnRenamed("certificate_status_clean",     "certificate_status")
    .withColumnRenamed("upload_date_parsed",           "upload_date")
    .withColumnRenamed("verification_date_parsed",     "verification_date")
)

# 6.3 Add derived columns to Silver for Gold layer KPIs
df_silver_batch = df_silver_batch \
    .withColumn(
        "days_to_verify",
        F.datediff(F.col("verification_date"), F.col("upload_date"))
        # NULL for Pending rows — expected, not an error
    ) \
    .withColumn(
        "upload_month",
        F.date_format(F.col("upload_date"), "yyyy-MM")
        # NULL for Pending rows — expected, not an error
    ) \
    .withColumn(
        "is_verified",
        F.when(F.col("certificate_status") == "Verified", True).otherwise(False)
    )

# 6.4 Merge constraint failures + duplicates into one quarantine batch
# Tag duplicates with their own error label so they are distinguishable in quarantine
df_duplicates_tagged = df_duplicates.withColumn(
    "_failed_constraints",
    F.array(F.lit("ERR: Duplicate employee_id + course_id combination"))
)

# Union both quarantine streams into a single quarantine table
df_quarantine_batch = df_quarantine_constraints.unionByName(df_duplicates_tagged)


# COMMAND ----------
# STEP 7: Diagnostics — Confirm the split before writing

silver_count      = df_silver_batch.count()
quarantine_count  = df_quarantine_batch.count()
bronze_count      = df_bronze.count()

print(f"\n{'='*50}")
print(f"  Bronze  (total)          : {bronze_count}")
print(f"  Silver  (clean)          : {silver_count}")
print(f"  Quarantine (constraints) : {df_quarantine_constraints.count()}")
print(f"  Quarantine (duplicates)  : {duplicate_count}")
print(f"  Quarantine (total)       : {quarantine_count}")
print(f"  Accounted for            : {silver_count + quarantine_count} / {bronze_count}")
print(f"{'='*50}\n")

# Preview quarantine error breakdown
print("Quarantine error distribution:")
df_quarantine_batch \
    .select(F.explode(F.col("_failed_constraints")).alias("error")) \
    .groupBy("error") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(truncate=False)


# COMMAND ----------
# STEP 8: Write to Target Delta Tables

# 8.1 Write clean records to Silver layer
silver_table_target = "hackathon_ltm.silver.sp_certification"

df_silver_batch.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table_target)

print(f"✅ Silver table written → {silver_table_target} ({silver_count} rows)")

# 8.2 Write all quarantine records (constraint failures + duplicates) to Quarantine layer
quarantine_table_target = "hackathon_ltm.quarantine.sp_certification"

df_quarantine_batch.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(quarantine_table_target)

print(f"✅ Quarantine table written → {quarantine_table_target} ({quarantine_count} rows)")
print(f"\nPipeline complete. Clean: {silver_table_target} | Quarantined: {quarantine_table_target}")

Target database environments ('silver' and 'quarantine') verified.
Bronze records ingested: 500
Duplicate rows detected and separated: 100

  Bronze  (total)          : 500
  Silver  (clean)          : 400
  Quarantine (constraints) : 0
  Quarantine (duplicates)  : 100
  Quarantine (total)       : 100
  Accounted for            : 500 / 500

Quarantine error distribution:
+--------------------------------------------------+-----+
|error                                             |count|
+--------------------------------------------------+-----+
|ERR: Duplicate employee_id + course_id combination|100  |
+--------------------------------------------------+-----+

✅ Silver table written → hackathon_ltm.silver.sp_certification (400 rows)
✅ Quarantine table written → hackathon_ltm.quarantine.sp_certification (100 rows)

Pipeline complete. Clean: hackathon_ltm.silver.sp_certification | Quarantined: hackathon_ltm.quarantine.sp_certification
